# 02 - TF-IDF Phase 1 Benchmark

This notebook is the official Phase 1 benchmark artifact for the TF-IDF extractive baseline.

Objectives:
- evaluate the TF-IDF baseline on the processed VietNews dataset under the frozen Phase 0 protocol
- compare multiple `top_k` settings on the same fixed subset
- quantify uncertainty beyond simple means
- export benchmark artifacts with strong reproducibility metadata for thesis reporting


## 1. Environment Setup

This notebook is intentionally restricted to `data/processed/vietnews/`.
It does not fall back to raw files and does not perform API smoke tests, so the benchmark remains protocol-aligned and reproducible.

In [ ]:
from __future__ import annotations

import json
import platform
import random
import subprocess
import sys
import time
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / "backend").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "backend").exists():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Repository root not found. Open the notebook from the repo root or notebooks/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from app.services.input import process_from_text
from app.services.summarization.tfidf_summarizer import summarize_with_tfidf
from evaluation.evaluator import Evaluator

pd.set_option("display.max_colwidth", 240)
evaluator = Evaluator(use_stemmer=False)


def safe_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


def safe_git_commit(repo_root: Path) -> str | None:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_root,
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except Exception:
        return None


environment_snapshot = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "rouge_score_version": safe_version("rouge-score"),
    "pydantic_version": safe_version("pydantic"),
    "git_commit": safe_git_commit(PROJECT_ROOT),
}

print("PROJECT_ROOT:", PROJECT_ROOT)
environment_snapshot


## 2. Configuration

Record these settings explicitly when transferring results into the thesis report.

In [ ]:
NOTEBOOK_SCHEMA_VERSION = "tfidf_phase1_benchmark_v2"
PROTOCOL_VERSION_EXPECTED = "phase0_v1"
TARGET_SPLIT = "validation"  # validation / test
SEED = 42
BOOTSTRAP_SAMPLES = 300
TOP_K_CANDIDATES = [2, 3, 4, 5]
SUBSET_LIMIT = 200
ARTICLE_CHAR_THRESHOLD = 1200  # set to None to use all eligible samples

random.seed(SEED)
np.random.seed(SEED)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "vietnews"
MANIFEST_PATH = PROCESSED_DIR / "dataset_manifest.json"
SPLIT_PATH = PROCESSED_DIR / f"{TARGET_SPLIT}.jsonl"

config_snapshot = {
    "notebook_schema_version": NOTEBOOK_SCHEMA_VERSION,
    "protocol_version_expected": PROTOCOL_VERSION_EXPECTED,
    "target_split": TARGET_SPLIT,
    "seed": SEED,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "top_k_candidates": TOP_K_CANDIDATES,
    "subset_limit": SUBSET_LIMIT,
    "article_char_threshold": ARTICLE_CHAR_THRESHOLD,
}
config_snapshot


## 3. Load The Processed Dataset And Freeze The Evaluation Subset

This step reads only the processed split and verifies the manifest protocol version before benchmarking.

In [ ]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Processed split not found: {SPLIT_PATH}")

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

protocol_version = manifest.get("protocol_version")
if protocol_version != PROTOCOL_VERSION_EXPECTED:
    raise RuntimeError(
        f"Protocol version mismatch: expected {PROTOCOL_VERSION_EXPECTED!r}, got {protocol_version!r}"
    )

rows: list[dict] = []
with SPLIT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("The processed split is empty.")

required_cols = ["guid", "title", "article", "reference_summary", "meta"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in the processed split: {missing_cols}")

df["article_char_len"] = df["meta"].map(lambda m: (m or {}).get("article_char_len", 0))
df["reference_char_len"] = df["meta"].map(lambda m: (m or {}).get("reference_summary_char_len", 0))
df = df[df["article"].fillna("").str.strip() != ""].copy()
df = df[df["reference_summary"].fillna("").str.strip() != ""].copy()

if ARTICLE_CHAR_THRESHOLD is not None:
    df = df[df["article_char_len"] >= ARTICLE_CHAR_THRESHOLD].copy()

subset_df = df.sample(frac=1.0, random_state=SEED).head(SUBSET_LIMIT).copy()
subset_df.reset_index(drop=True, inplace=True)

subset_overview = {
    "loaded_rows": int(len(rows)),
    "eligible_rows_after_filter": int(len(df)),
    "subset_rows": int(len(subset_df)),
    "target_split": TARGET_SPLIT,
    "article_char_threshold": ARTICLE_CHAR_THRESHOLD,
}
subset_overview


## 4. Qualitative Inspection On One Frozen Example

This section is used only to explain how TF-IDF ranks and selects sentences.

In [ ]:
demo_row = subset_df.iloc[0].to_dict()
demo_processed = process_from_text(str(demo_row.get("article", "")))

inspect_rows: list[dict] = []
for top_k in TOP_K_CANDIDATES:
    selected_sentences, engine_meta = summarize_with_tfidf(
        demo_processed.sentences,
        max_sentences=top_k,
        ratio=None,
    )
    selected_indices = set(engine_meta.get("selected_indices", []))
    for rank, item in enumerate(
        sorted(engine_meta.get("sentence_scores", []), key=lambda x: x["score"], reverse=True),
        start=1,
    ):
        idx = item["index"]
        inspect_rows.append(
            {
                "top_k": top_k,
                "rank_by_score": rank,
                "sentence_index": idx,
                "score": item["score"],
                "is_selected": idx in selected_indices,
                "sentence": demo_processed.sentences[idx],
            }
        )

inspect_df = pd.DataFrame(inspect_rows)
inspect_df.head(20)


## 5. Run The Benchmark

Latency is measured only around `summarize_with_tfidf(...)`, not around `process_from_text(...)`.
Therefore, the latency reported below is **summarizer-core latency**, not end-to-end pipeline latency.

In [ ]:
def summarize_article(article: str, top_k: int) -> tuple[str, dict, float, str]:
    processed = process_from_text(article)
    t0 = time.perf_counter()
    selected_sentences, engine_meta = summarize_with_tfidf(
        processed.sentences,
        max_sentences=top_k,
        ratio=None,
    )
    summarizer_core_latency_sec = time.perf_counter() - t0
    summary = " ".join(sentence.strip() for sentence in selected_sentences if sentence.strip())
    return summary, engine_meta, summarizer_core_latency_sec, processed.cleaned_text


records: list[dict] = []
for top_k in TOP_K_CANDIDATES:
    for row in subset_df.to_dict(orient="records"):
        article = str(row.get("article", ""))
        reference = str(row.get("reference_summary", ""))
        if not article.strip() or not reference.strip():
            continue

        predicted_summary, engine_meta, summarizer_core_latency_sec, cleaned_article = summarize_article(article, top_k)
        bundle = evaluator.evaluate_one(
            source_text=cleaned_article,
            reference_summary=reference,
            predicted_summary=predicted_summary,
            latency_sec=summarizer_core_latency_sec,
            extra={
                "guid": row.get("guid"),
                "top_k": top_k,
                "article_char_len": row.get("article_char_len"),
                "reference_char_len": row.get("reference_char_len"),
                "engine": engine_meta.get("engine", "tfidf"),
            },
        )
        rec = bundle.as_dict()
        rec["summarizer_core_latency_sec"] = rec.pop("latency_sec")
        records.append(rec)

bench_df = pd.DataFrame(records)
if bench_df.empty:
    raise RuntimeError("No valid samples are available for benchmarking.")
bench_df.head(3)


## 6. Aggregate Means, Standard Deviations, And Bootstrap Confidence Intervals

Means alone are not sufficient when comparing nearby configurations. This section adds per-metric standard deviations and simple bootstrap confidence intervals.

In [ ]:
METRIC_COLUMNS = [
    "rouge1_f",
    "rouge2_f",
    "rougeL_f",
    "summarizer_core_latency_sec",
    "compression_ratio",
    "repetition_rate",
]


def bootstrap_ci_mean(values: pd.Series, n_boot: int = 300, seed: int = 42) -> tuple[float, float]:
    arr = values.dropna().to_numpy(dtype=float)
    if len(arr) == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    boot_means = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        boot_means.append(float(sample.mean()))
    low, high = np.percentile(boot_means, [2.5, 97.5])
    return float(low), float(high)


summary_rows: list[dict] = []
for top_k, group in bench_df.groupby("top_k"):
    row = {
        "top_k": int(top_k),
        "n": int(len(group)),
    }
    for metric in METRIC_COLUMNS:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_std"] = float(group[metric].std(ddof=1)) if len(group) > 1 else 0.0
        ci_low, ci_high = bootstrap_ci_mean(group[metric], n_boot=BOOTSTRAP_SAMPLES, seed=SEED + int(top_k))
        row[f"{metric}_ci95_low"] = ci_low
        row[f"{metric}_ci95_high"] = ci_high
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values("top_k").reset_index(drop=True)
summary_df


## 7. Multi-Metric Recommendation View

Instead of choosing `best_top_k` from a single metric, we derive a simple weighted rank across quality and efficiency metrics.
This remains a heuristic, but it is more defensible than selecting only by ROUGE-1.

In [ ]:
selection_view_df = summary_df.copy()

selection_view_df["rank_rouge1"] = selection_view_df["rouge1_f_mean"].rank(ascending=False, method="min")
selection_view_df["rank_rouge2"] = selection_view_df["rouge2_f_mean"].rank(ascending=False, method="min")
selection_view_df["rank_rougeL"] = selection_view_df["rougeL_f_mean"].rank(ascending=False, method="min")
selection_view_df["rank_compression"] = selection_view_df["compression_ratio_mean"].rank(ascending=True, method="min")
selection_view_df["rank_repetition"] = selection_view_df["repetition_rate_mean"].rank(ascending=True, method="min")
selection_view_df["rank_latency"] = selection_view_df["summarizer_core_latency_sec_mean"].rank(ascending=True, method="min")

selection_view_df["weighted_rank_score"] = (
    0.30 * selection_view_df["rank_rouge1"]
    + 0.20 * selection_view_df["rank_rouge2"]
    + 0.25 * selection_view_df["rank_rougeL"]
    + 0.15 * selection_view_df["rank_compression"]
    + 0.05 * selection_view_df["rank_repetition"]
    + 0.05 * selection_view_df["rank_latency"]
)

selection_view_df = selection_view_df.sort_values(["weighted_rank_score", "rouge1_f_mean"], ascending=[True, False]).reset_index(drop=True)
recommended_row = selection_view_df.iloc[0].to_dict()

best_by_metric = {
    "rouge1_f": int(summary_df.sort_values("rouge1_f_mean", ascending=False).iloc[0]["top_k"]),
    "rouge2_f": int(summary_df.sort_values("rouge2_f_mean", ascending=False).iloc[0]["top_k"]),
    "rougeL_f": int(summary_df.sort_values("rougeL_f_mean", ascending=False).iloc[0]["top_k"]),
    "compression_ratio": int(summary_df.sort_values("compression_ratio_mean", ascending=True).iloc[0]["top_k"]),
    "repetition_rate": int(summary_df.sort_values("repetition_rate_mean", ascending=True).iloc[0]["top_k"]),
    "summarizer_core_latency_sec": int(summary_df.sort_values("summarizer_core_latency_sec_mean", ascending=True).iloc[0]["top_k"]),
}

selection_view_df[[
    "top_k",
    "weighted_rank_score",
    "rouge1_f_mean",
    "rouge2_f_mean",
    "rougeL_f_mean",
    "compression_ratio_mean",
    "repetition_rate_mean",
    "summarizer_core_latency_sec_mean",
]]


## 8. Conclusions

The notebook reports both per-metric winners and a multi-metric recommendation. This is more appropriate for thesis reporting when there is a genuine trade-off between ROUGE and summary compactness.

In [ ]:
conclusion = {
    "recommended_top_k_by_weighted_rank": int(recommended_row["top_k"]),
    "recommended_row": recommended_row,
    "best_by_metric": best_by_metric,
    "selection_note": (
        "The recommended top_k is selected by a weighted rank across ROUGE-1, ROUGE-2, ROUGE-L, "
        "compression ratio, repetition rate, and summarizer-core latency. "
        "This is still a heuristic and should be interpreted together with the full summary table."
    ),
}
conclusion


## 9. Save Benchmark Artifacts

Export the aggregate table, the per-sample detail table, and a JSON report for use in the experiment chapter.
The report schema now includes stronger metadata so future artifacts can be traced back to a specific notebook version and code state.

In [ ]:
out_dir = PROJECT_ROOT / "notebooks" / "results"
out_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_path = out_dir / f"tfidf_phase1_topk_summary_{ts}.csv"
detail_path = out_dir / f"tfidf_phase1_topk_detail_{ts}.csv"
report_path = out_dir / f"tfidf_phase1_topk_report_{ts}.json"

summary_df.to_csv(summary_path, index=False, encoding="utf-8")
bench_df.to_csv(detail_path, index=False, encoding="utf-8")

report_payload = {
    "report_schema_version": NOTEBOOK_SCHEMA_VERSION,
    "notebook": "02_tfidf_experiment.ipynb",
    "protocol_version": protocol_version,
    "target_split": TARGET_SPLIT,
    "seed": SEED,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "top_k_candidates": TOP_K_CANDIDATES,
    "article_char_threshold": ARTICLE_CHAR_THRESHOLD,
    "subset_limit": SUBSET_LIMIT,
    "subset_rows": int(len(subset_df)),
    "subset_guid_sample": subset_df["guid"].head(20).tolist(),
    "environment": environment_snapshot,
    "config_snapshot": config_snapshot,
    "recommended_top_k_by_weighted_rank": int(recommended_row["top_k"]),
    "best_by_metric": best_by_metric,
    "artifact_note": (
        "Older TF-IDF artifacts in the repository may have been generated from a previous notebook schema. "
        "Use report_schema_version and config_snapshot to audit compatibility."
    ),
    "summary_csv": str(summary_path),
    "detail_csv": str(detail_path),
    "manifest_path": str(MANIFEST_PATH),
    "split_path": str(SPLIT_PATH),
}

report_path.write_text(json.dumps(report_payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:")
print(summary_path)
print(detail_path)
print(report_path)
report_payload


## 10. How To Read The Results

- `summary_df`: aggregate means, standard deviations, and bootstrap confidence intervals for each `top_k`.
- `bench_df`: per-sample detail table, suitable for later error analysis.
- `selection_view_df`: a multi-metric recommendation table that makes trade-offs explicit.
- `report_payload`: benchmark metadata aligned with the frozen Phase 0 protocol and strengthened for thesis reproducibility.
